# NB 2.1 &mdash; Solucions dels exercicis

**MP 5134** &mdash; UT2 · *Dades: pingüins de l'arxipèlag Palmer*

---

Solucionari dels cinc exercicis de la secció 7 del
[NB 2.1](NB_2_1_regressio_lineal_simple.ipynb) i orientacions per al debat de la
secció 8.

La primera cel·la reconstrueix l'estat del notebook original perquè aquest
solucionari es pugui executar sol.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"
df = pd.read_csv(URL_DADES)
data = df[["flipper_length_mm", "body_mass_g"]].dropna()

X = data[["flipper_length_mm"]]
y = data["body_mass_g"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)

def predir_amb_recta(aleta, pendent, ordenada):
    return ordenada + pendent * aleta

def error_mitja(y_real, y_predit):
    return (y_predit - y_real).abs().mean()

print(f"Model de referència del notebook: pendent {model.coef_[0]:.2f}, ordenada {model.intercept_:.0f}")

## Exercici 1

> Canvia `PENDENT_B` i `ORDENADA_B` fins a aconseguir un error mitjà sobre
> l'entrenament més petit que el de la recta A (344 g).

**Resposta.** La manera més ràpida és aplicar el truc de la secció 4.2: triar un
pendent i calcular l'ordenada perquè la recta passi pel centre del núvol. Però
també es pot fer a les palpentes, tocant els dos números.

In [ ]:
for pendent, ordenada in [(60, -7800), (55, -6850), (50, -5850)]:
    pred = predir_amb_recta(X_train["flipper_length_mm"], pendent, ordenada)
    print(f"pendent {pendent}, ordenada {ordenada}: error mitjà {error_mitja(y_train, pred):.1f} g")

print(f"scikit-learn:                     error mitjà {error_mitja(y_train, model.predict(X_train)):.1f} g")

Amb pendent 50 i ordenada &minus;5.850 ja s'arriba a 318,5 g, i aquí hi ha una
sorpresa que val la pena aprofitar a classe: **és una mica millor que la recta de
scikit-learn** (318,9 g).

No és cap error. Com explica la secció 6.4, `LinearRegression` no busca la recta
amb el MAE més petit, sinó la que té més petits **els errors al quadrat**. Si
algú troba una recta amb menys MAE, ha trobat una recta millor *segons el MAE*, i
segurament pitjor *segons el RMSE*. Es pot comprovar:

In [ ]:
pred_ma = predir_amb_recta(X_train["flipper_length_mm"], 50, -5850)
pred_sk = model.predict(X_train)

print(f"Recta a mà:    MAE {mean_absolute_error(y_train, pred_ma):.1f} g   RMSE {np.sqrt(mean_squared_error(y_train, pred_ma)):.2f} g")
print(f"scikit-learn:  MAE {mean_absolute_error(y_train, pred_sk):.1f} g   RMSE {np.sqrt(mean_squared_error(y_train, pred_sk)):.2f} g")

*Per a la correcció:* l'objectiu no és trobar els decimals, és que l'alumnat
entengui que **ajustar un model és ajustar números fins que un error sigui mínim**.
Si algú arriba a menys de 330 g amb qualsevol estratègia, l'exercici està resolt.
Qui descobreixi que pot guanyar scikit-learn en MAE es mereix que li expliquis el
perquè davant de tothom.

## Exercici 2

> Repeteix les seccions 3, 5 i 6 fent servir `bill_depth_mm`. Quin signe té el
> pendent? Quin MAE i quin R2 surten?

In [ ]:
dades_bec = df[["bill_depth_mm", "body_mass_g"]].dropna()

plt.figure(figsize=(7, 5))
for especie, grup in df.groupby("species"):
    plt.scatter(grup["bill_depth_mm"], grup["body_mass_g"], alpha=0.5, label=especie)
plt.xlabel("Gruix del bec (mm)")
plt.ylabel("Massa (g)")
plt.legend()
plt.show()

Xb = dades_bec[["bill_depth_mm"]]
yb = dades_bec["body_mass_g"]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.2, random_state=42)

model_bec = LinearRegression().fit(Xb_train, yb_train)
pred_bec = model_bec.predict(Xb_test)

print(f"Pendent: {model_bec.coef_[0]:.0f} g per mm de gruix")
print(f"MAE:     {mean_absolute_error(yb_test, pred_bec):.0f} g")
print(f"R2:      {r2_score(yb_test, pred_bec):.2f}")

**Resposta.** El pendent és **negatiu**: uns &minus;200 g per mil·límetre. El model
diu que *com més gruixut és el bec, menys pesa el pingüí*, cosa que sona absurda.
El MAE puja a uns 540 g i el R2 cau a 0,21, gairebé com dir la mitjana.

La clau és al dibuix, que aquí l'hem acolorit per espècie. **Els Gentoo són els
pingüins més pesants i tenen el bec més prim.** Dins de cada espècie, en canvi, el
bec més gruixut sí que va amb més pes. Quan ho barregem tot, el grup dels Gentoo
arrossega la recta cap avall.

És la **paradoxa de Simpson**: una relació que té un signe dins de cada grup i el
signe contrari quan ajuntes els grups. Dins de cada espècie, el gruix del bec i la
massa van clarament junts; amb totes barrejades, la recta surt al revés. A la UT3
la tornaràs a trobar entre les dues mesures del bec.

*Per a la correcció:* no cal que l'alumnat conegui el nom de la paradoxa. Sí que
cal que noti que el signe és estrany i que en busqui l'explicació al dibuix. Qui
proposi *"caldria tenir en compte l'espècie"* ha arribat exactament a la UT3.

## Exercici 3

> Calcula la massa predita per a un pingüí amb una aleta de 195 mm fent servir
> només `coef_` i `intercept_`.

In [ ]:
a_ma = model.intercept_ + model.coef_[0] * 195
amb_predict = model.predict(pd.DataFrame({"flipper_length_mm": [195]}))[0]

print(f"A mà:         {a_ma:.1f} g")
print(f"Amb predict(): {amb_predict:.1f} g")

**Resposta.** Uns **3.904 g**, i els dos càlculs coincideixen fins a l'últim
decimal. És la confirmació pràctica que un model lineal entrenat no amaga res
més que aquests dos números.

*Per a la correcció:* l'error típic és oblidar que `coef_` és una llista i
escriure `model.coef_ * 195`, que dóna un array d'un element en lloc d'un número.
Funciona igualment, però val la pena comentar-ho.

## Exercici 4

> Troba el pingüí del conjunt de prova amb l'error més gran.

In [ ]:
errors = pd.Series(model.predict(X_test) - y_test.values, index=X_test.index)
index_pitjor = errors.abs().idxmax()

print(f"Error: {errors[index_pitjor]:.0f} g")
df.loc[index_pitjor]

**Resposta.** És un **mascle Adelie** amb una aleta de 191 mm que pesa 4.600 g. El
model li calcula uns 3.700 g: s'equivoca gairebé **900 g per sota**.

El model no hi podia fer gaire. Amb una aleta d'uns 190 mm, la majoria de
pingüins pesen entre 3.400 i 3.900 g, i aquest és justament el més pesant de tots.
És un mascle especialment corpulent per a la seva aleta, i el model no sap que és
mascle perquè no li ho hem dit. Si mires els següents pitjors casos, hi trobaràs
sobretot mascles.

*Per a la correcció:* aquest exercici connecta amb la idea de la secció 2, el
*gruix del núvol*. Els errors grans no són fallades del codi, són informació que
el model no té. Qui proposi afegir el sexe com a variable ha entès el missatge.

## Exercici 5

> Calcula el MAE i el RMSE sobre l'entrenament i sobre la prova. S'assemblen?

In [ ]:
for nom, Xc, yc in [("entrenament", X_train, y_train), ("prova", X_test, y_test)]:
    pred = model.predict(Xc)
    print(f"{nom:>12}:  MAE {mean_absolute_error(yc, pred):.0f} g   RMSE {np.sqrt(mean_squared_error(yc, pred)):.0f} g")

**Resposta.** S'assemblen molt, i fins i tot la prova surt **una mica millor** que
l'entrenament (288 g contra 319 g de MAE).

La conclusió és que **el model no ha memoritzat res**: s'equivoca igual amb
pingüins que ha vist i amb pingüins que no. Una recta no té prou llibertat per
memoritzar. Que la prova surti una mica millor és pura sort de la partició: els
69 pingüins del test són, de mitjana, una mica més fàcils. Ja ho vam veure a la
solució de l'exercici 2 del NB 1.2.

*Per a la correcció:* la resposta que cal buscar és *"s'assemblen, per tant el
model generalitza bé"*. Al NB 2.3 veuran exactament el cas contrari, i convé que
arribin amb aquesta comparació ja feta.

## Orientacions per al debat: les vendes setmanals

No hi ha una resposta única, però sí arguments que convé que surtin:

- **El MAE** és el més fàcil d'explicar a qui fa les comandes: *"de mitjana ens
  equivoquem en 40 unitats per setmana"*. El problema és que barreja productes molt
  diferents: 40 unitats és un desastre per a un producte que en ven 50 i res per a
  un que en ven 3.000.
- **La desviació percentual** resol aquest problema i permet comparar productes,
  però es torna boja amb els productes que venen molt poc. Si un producte ven 1
  unitat i en prediem 3, l'error és del 200%. I si alguna setmana en ven 0, no es
  pot ni calcular.
- **El RMSE** és el que convé si els errors grans són especialment cars: per
  exemple, quedar-se sense estoc del producte estrella en plena campanya de Nadal.
- **El R2** és el menys útil per a la direcció: no diu res en unitats ni en diners.

La pregunta de l'estoc és la més interessant. Quedar-se curt i passar-se no costen
el mateix: una ruptura d'estoc fa perdre vendes i clients, i un excedent de
iogurts es fa malbé, però un excedent de detergent es ven la setmana següent.
**Cap de les mètriques que hem vist distingeix entre errors per dalt i per baix**,
i en un projecte real això es resol amb mètriques fetes a mida. Si el grup hi
arriba, el debat ha anat molt bé.